In [ ]:
"""Day 25 / Session 3 - Lab Assignment 2: BERT Inputs & Pooling - WordPiece Tokenizer + [CLS]/Mean Pooling."""
import numpy as np


def wordpiece_tokenize(word, vocab, unk_token="[UNK]", max_chars=100):
    if len(word) > max_chars:
        return [unk_token]
    pieces = []
    start = 0
    while start < len(word):
        end = len(word)
        current = None
        while start < end:                                   # shrink the end until the substring is in the vocab
            piece = word[start:end]
            if start > 0:
                piece = "##" + piece                         # continuation pieces carry the ## prefix
            if piece in vocab:
                current = piece
                break
            end -= 1
        if current is None:
            return [unk_token]                               # nothing matched: the whole word is unknown
        pieces.append(current)
        start = end
    return pieces


def tokenize_text(text, vocab, unk_token="[UNK]"):
    tokens = []
    for word in str(text).split():
        tokens.extend(wordpiece_tokenize(word, vocab, unk_token))
    return tokens


def cls_pool(hidden_states):
    return np.asarray(hidden_states, dtype=float)[0]         # the [CLS] vector sits at position 0


def masked_mean_pool(hidden_states, attention_mask):
    H = np.asarray(hidden_states, dtype=float)
    keep = np.asarray(attention_mask) == 1
    if not keep.any():
        return np.zeros(H.shape[1])
    return H[keep].mean(axis=0)                              # padding rows are ignored
